In [18]:
#from google.colab import files
#uploaded = files.upload()

In [19]:
import os
import zipfile
import numpy as np
from PIL import Image

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler
from imblearn.pipeline import Pipeline


from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier

from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    ConfusionMatrixDisplay,
    classification_report,
)

In [20]:
# Extracted zipped file, and then flatten the images into numpy arrays:
with zipfile.ZipFile("sad_cleaned-20260925T025510Z-1-001.zip", "r") as z:
  z.extractall("sad_cleaned")

x=[];y=[]

classes = ['sad_cleaned/sad_cleaned_test', 'sad_cleaned/sad_cleaned_train']

for label, folder in enumerate(classes):
  path = os.path.join("sad_cleaned", folder)

  for filename in os.listdir(path):
    img_path = os.path.join(path, filename)

    try:
      img = Image.open(img_path)
      img = img.resize((64, 64))

      features = np.array(img).flatten()

      x.append(features)
      y.append(label)
    except Exception as e:
      print(f"Skipped {filename}: {e}")

x=np.array(x);y=np.array(y)
print(x.shape);print(y.shape)

(1245, 12288)
(1245,)


In [ ]:
# Train/Test sets
X_train, X_test, y_train, y_test = train_test_split(x, y,
                                                    test_size=0.2,
                                                    random_state=42,
                                                    stratify=y
                                                    )

In [ ]:
# Data Augmentation?

In [21]:
from sklearn.ensemble import RandomForestClassifier
pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('classifier', RandomForestClassifier(max_depth=100, n_estimators=10))
])
pipe.fit(X_train, y_train)
y_pred = pipe.predict(X_test)
print(classification_report(y_test, y_pred, target_names=['Sad', 'Not Sad']))


              precision    recall  f1-score   support

         Sad       0.69      0.88      0.77       173
     Not Sad       0.28      0.11      0.15        76

    accuracy                           0.64       249
   macro avg       0.48      0.49      0.46       249
weighted avg       0.56      0.64      0.58       249



In [23]:
from sklearn.tree import DecisionTreeClassifier
model = DecisionTreeClassifier(
    criterion="gini",
    max_depth=100,
    random_state=42
)

model.fit(X_train, y_train)

predictions = model.predict(X_test)
print(classification_report(y_test, predictions, target_names=['Sad', 'Not Sad']))

              precision    recall  f1-score   support

         Sad       0.72      0.72      0.72       173
     Not Sad       0.36      0.36      0.36        76

    accuracy                           0.61       249
   macro avg       0.54      0.54      0.54       249
weighted avg       0.61      0.61      0.61       249



In [24]:
# Model after Data Augmentation. So, start from the begining x,y numpy arrays.
from torchvision import transforms

augment = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.RandomResizedCrop(64, scale=(0.9, 1.1)),
])
x=[];y=[]

classes = ['sad_cleaned/sad_cleaned_test', 'sad_cleaned/sad_cleaned_train']

for label, folder in enumerate(classes):
  path = os.path.join("sad_cleaned", folder)

  for filename in os.listdir(path):
    img_path = os.path.join(path, filename)

    try:
      img = Image.open(img_path)
      img = img.resize((64, 64))

      features = np.array(img).flatten()

      x.append(features)
      y.append(label)

      auger = augment(img)

      x.append(np.array(auger.resize((64,64))).flatten())
      y.append(label)
    except Exception as e:
      print(f"Skipped {filename}: {e}")

x=np.array(x);y=np.array(y)
print(x.shape);print(y.shape)


(2490, 12288)
(2490,)


In [25]:
pipe.fit(X_train, y_train)
y_pred = pipe.predict(X_test)
print(classification_report(y_test, y_pred, target_names=['Sad', 'Not Sad']))

              precision    recall  f1-score   support

         Sad       0.71      0.91      0.80       173
     Not Sad       0.46      0.17      0.25        76

    accuracy                           0.69       249
   macro avg       0.59      0.54      0.53       249
weighted avg       0.64      0.69      0.63       249



In [26]:
model.fit(X_train, y_train)

predictions = model.predict(X_test)
print(classification_report(y_test, predictions, target_names=['Sad', 'Not Sad']))

              precision    recall  f1-score   support

         Sad       0.72      0.72      0.72       173
     Not Sad       0.36      0.36      0.36        76

    accuracy                           0.61       249
   macro avg       0.54      0.54      0.54       249
weighted avg       0.61      0.61      0.61       249



## Notes:

\[ model performance $\implies$ augmentation $\implies$ good data \]